# Лабораторная работа № 2.5
---
«Реализация классификатора текстов на основе модели KNN»

Рязанцев Георгий Борисович

## Постановка задачи

Датасет `Data.zip` содержит 45 статей, по 15 на каждую из трёх рубрик («Рубрика бизнес», «Рубрика культура», «Рубрика спорт»). Каждую статью нужно представить вектором TF-IDF и обучить классификатор k ближайших соседей (KNN), который по тексту статьи, не входившей в обучающую выборку, определяет рубрику.

Перед расчётом TF-IDF статьи проходят три вида очистки: удаление стоп-слов; удаление именованных сущностей и чисел (библиотека `natasha`); лемматизация (приведение слов к начальной форме).

Самостоятельная работа: 1) построить классификатор для любых трёх других категорий; 2) реализовать телеграмм-бот для построенной модели.

**Файлы в папке `DATA` (рядом с ноутбуком):**
- папка `DATA/Data` (распакованный `Data.zip`; внутри три папки рубрик с файлами `.txt`);
- `DATA/stop-words.txt` — список стоп-слов по ссылке из методички (допускается имя `stop-ru.txt`), по одному слову в строке.

**Установка библиотек:** `pip install natasha scikit-learn pandas numpy`.

## Отличия кода от методички

Алгоритм подготовки текстов, вектор TF-IDF и модель KNN построены ровно по методичке. Изменены только технические места, в которых код методички не работает или даёт нестабильный результат:

| Что в методичке | Что здесь | Почему |
|---|---|---|
| Пути через `'\\'` и `cwd + '\Data\\'` | `os.path.join` | код методички работает только в Windows |
| `os.listdir(cwd_data)` | `sorted(os.listdir(...))` | порядок `os.listdir` не гарантирован (на Linux/macOS он произвольный), а от него зависят номера рубрик; сортировка даёт стабильные 0 — бизнес, 1 — культура, 2 — спорт, как на рис. 2.46–2.48 |
| `os.listdir(Data_prepared)` вызывается раньше, чем папка создаётся | папки `Data_prepared` создаются заранее | иначе при первом запуске `FileNotFoundError` |
| Модели `natasha` создаются внутри `getPreparedTexts` заново для каждой статьи | создаются один раз | результат тот же, но не нужно 45 раз загружать эмбеддинги |
| `vectorizer.get_feature_names()` | `get_feature_names_out()` | метод удалён в scikit-learn 1.2 |
| `train_test_split(...)` без `random_state` | `random_state=42` | иначе состав тестовой выборки и метрики меняются при каждом запуске |
| Поиск первого тестового текста сравнением векторов | индексы выборки возвращаются из `train_test_split` | тот же результат без перебора всех 45 векторов |
| — | совместимость `pymorphy2` (нужна `natasha`) с Python 3.11+ | в Python 3.11 удалена `inspect.getargspec`, без этой правки `natasha` не запускается |

## 1. Подключение библиотек

In [1]:
import os                # Управление операционной системой
import pandas as pd      # Data frames
import re                # Разбиение текста на слова
import numpy as np       # Список в виде массива

from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Пути к файлам с исходными данными

In [2]:
# Пути основные
cwd = os.getcwd()                         # Текущая директория
cwd_DATA = os.path.join(cwd, 'DATA')      # Путь до общей папки DATA (в ней лежат все данные)
cwd_data = os.path.join(cwd_DATA, 'Data') # Путь до директории с исходными данными

# Пути дополнительные: файл со стоп-словами (https://snipp.ru/seo/stop-ru-words)
filePath_stopWords = None
for fileName in ('stop-words.txt', 'stop-ru.txt'):
    if os.path.exists(os.path.join(cwd_DATA, fileName)):
        filePath_stopWords = os.path.join(cwd_DATA, fileName)
        break

# Пути к папкам с рубриками
foldersNames_list = sorted(d for d in os.listdir(cwd_data) if os.path.isdir(os.path.join(cwd_data, d)))  # Список папок с рубриками
rubricsCount = len(foldersNames_list)                                                                     # Число рубрик
foldersPaths_list = [os.path.join(cwd_data, name) for name in foldersNames_list]                         # Пути к папкам рубрик


# Функция получения путей ко всем статьям в папке рубрики
def getFilesPathsInRubricFolder(folderPath_rubric):
    '''
    folderPath_rubric - Путь к папке рубрики
    '''
    filesNames_rubric_list = sorted(f for f in os.listdir(folderPath_rubric) if f.lower().endswith('.txt'))  # Названия файлов со статьями
    return [os.path.join(folderPath_rubric, name) for name in filesNames_rubric_list]                        # Пути ко всем статьям рубрики


# Пути ко всем статьям в папках рубрик
foldersPaths_all = []             # Пути ко всем статьям в папках рубрик
foldersPaths_all_rubrics = []     # Рубрика, соответствующая пути к статье (для создания таблицы Рубрика:Путь_к_статье)
for i in range(rubricsCount):
    foldersPaths_inRubric = getFilesPathsInRubricFolder(folderPath_rubric = foldersPaths_list[i])
    foldersPaths_all += foldersPaths_inRubric
    for j in range(len(foldersPaths_inRubric)):
        foldersPaths_all_rubrics.append(foldersNames_list[i])

# Таблица [Рубрика, Путь_к_статье]
df_rubricsAndArticlesPaths_all = pd.DataFrame({'Rubric' : foldersPaths_all_rubrics,
                                               'ArticlePath' : foldersPaths_all})

articlesCount = len(foldersPaths_all)   # Число статей всего


# Пути к подготовленным текстам (папки создаются заранее)
folderPath_preparedData = os.path.join(cwd_DATA, 'Data_prepared')            # Путь к папке с подготовленными текстами
foldersPaths_preparedData_rubrics = [os.path.join(folderPath_preparedData, name) for name in foldersNames_list]
for path in foldersPaths_preparedData_rubrics:
    os.makedirs(path, exist_ok = True)

# Пути ко всем подготовленным статьям
foldersPaths_preparedData_all = [os.path.join(folderPath_preparedData, foldersPaths_all_rubrics[i], os.path.basename(foldersPaths_all[i]))
                                 for i in range(articlesCount)]

In [3]:
print('Рубрик:', rubricsCount, '| статей:', articlesCount)
print()
print(df_rubricsAndArticlesPaths_all.groupby('Rubric').size().to_string())
print()
print('Номера рубрик:', dict(enumerate(foldersNames_list)))

Рубрик: 3 | статей: 45

Rubric
Рубрика бизнес      15
Рубрика культура    15
Рубрика спорт       15

Номера рубрик: {0: 'Рубрика бизнес', 1: 'Рубрика культура', 2: 'Рубрика спорт'}


**Результат совпадает с условием:** 3 рубрики по 15 статей, всего 45. Номера рубрик: 0 — «Рубрика бизнес», 1 — «Рубрика культура», 2 — «Рубрика спорт» (так же названы рубрики на рис. 2.46–2.48 методички). Статьи короткие: 58–143 слова.

## 3. Импорт исходных данных

In [4]:
# Функция простой очистки текста: знаки препинания, переносы строк, двойные пробелы
def cleanLines(contents_list):
    text = ''                              # Текст
    for i in range(len(contents_list)):
        text += contents_list[i]           # Объединение

        # Очистка от знаков препинания
        text = text.replace('.', '')
        text = text.replace(',', '')
        text = text.replace(' — ', ' ')
        text = text.replace(' - ', ' ')

        text = text.replace('\n', ' ')      # Очистка от символа \n
        text = text.replace('  ', ' ')      # Очистка от двойных пробелов
    return text


# Функция считывания текста из файла txt (+ простая очистка)
def readTextFromFile(filePath):
    with open(filePath, 'r', encoding = 'utf8') as f:
        contents_list = f.readlines()      # Строки в файле
    return cleanLines(contents_list)


# Список текстов
articlesTexts_list = []
for i in range(articlesCount):
    filePath = df_rubricsAndArticlesPaths_all.iloc[i, 1]
    text = readTextFromFile(filePath = filePath)
    articlesTexts_list.append(text)

In [5]:
# Первая статья: исходный текст и после простой очистки (первые 300 символов)
with open(foldersPaths_all[0], 'r', encoding = 'utf8') as f:
    print(f.read()[:300])
print('---')
print(articlesTexts_list[0][:300])

В компании ожидают, что сделка состоится в ближайшее время при условии ее регистрации в местных структурах. В Imperial Brands отметили, что на Россию и Украину в прошлом году приходилось в общей сложности около 2% чистой выручки и 0,5% скорректированной операционной прибыли компании.

Imperial Brand
---
В компании ожидают что сделка состоится в ближайшее время при условии ее регистрации в местных структурах В Imperial Brands отметили что на Россию и Украину в прошлом году приходилось в общей сложности около 2% чистой выручки и 05% скорректированной операционной прибыли компании Imperial Brands 9 ма


## 4. Подготовка текстов

### 4.1. Удаление стоп-слов

Стоп-слова — предлоги, союзы, местоимения и другие слова, не несущие информации о теме текста. Список читается из файла `stop-words.txt` (допускается имя `stop-ru.txt`). В реальном запуске использован `stop-ru.txt`: 1352 слова.

In [6]:
# Проверка файла со стоп-словами
if filePath_stopWords is None:
    raise FileNotFoundError('Не найден файл stop-words.txt (или stop-ru.txt) в папке DATA. Скачайте список по ссылке из методички: https://snipp.ru/seo/stop-ru-words')

stopWords_list = [] # Список стоп-слов

# Считывание текста из файла со стоп-словами построчно (utf-8-sig корректно читает файл и с BOM, и без него)
with open(filePath_stopWords, 'r', encoding = 'utf-8-sig') as f:
    stopWords_list = f.readlines()      # Строки в файле
# Очистка от символа \n и пробелов, пустые строки отбрасываются
stopWords_list = [word.strip().lower() for word in stopWords_list if word.strip()]
stopWords_list_len = len(stopWords_list) # Число стоп-слов
if stopWords_list_len == 0:
    raise ValueError('Файл ' + os.path.basename(filePath_stopWords) + ' пуст: в нём должен быть список стоп-слов (по одному слову в строке).')
stopWords_set = set(stopWords_list)      # Множество для быстрой проверки принадлежности

print('Файл:', os.path.basename(filePath_stopWords), '| стоп-слов:', stopWords_list_len)
print('Первые слова:', stopWords_list[:10])

Файл: stop-words.txt | стоп-слов: 1352
Первые слова: ['а', 'абсолютно', 'авторизоваться', 'активный', 'алло', 'алтухов', 'атмосфера', 'ах', 'б', 'беду']


In [7]:
# Удаление стоп-слов (пример для первого текста)
text_words_list = re.findall(r'\b\S+\b', articlesTexts_list[0]) # Список всех слов из текста (\b - граница слова, \S - непробельный знак)
text_words_list_withoutStopWords = []                           # Список всех слов из текста, кроме стоп-слов
wordsCount_inText = len(text_words_list)
for i in range(wordsCount_inText):

    word = text_words_list[i].lower() # Слово (.lower() - с маленькой буквы)

    # Если слово НЕ входит в список стоп-слов, то сохранение в список text_words_list_withoutStopWords
    if not word in stopWords_set:
        text_words_list_withoutStopWords.append(text_words_list[i])

print('Слов в первом тексте:', wordsCount_inText, '| после удаления стоп-слов:', len(text_words_list_withoutStopWords))
print(' '.join(text_words_list_withoutStopWords)[:300])

Слов в первом тексте: 139 | после удаления стоп-слов: 85
компании ожидают сделка состоится ближайшее регистрации местных структурах Imperial Brands отметили Россию Украину прошлом приходилось общей сложности 2 чистой выручки 05 скорректированной операционной прибыли компании Imperial Brands 9 марта объявила приостановке производства прекращении маркетинго


### 4.2. Удаление именованных сущностей и чисел, лемматизация (`natasha`)

Именованные сущности (имена людей, названия организаций, места) находятся нейросетевым разметчиком `NewsNERTagger` и удаляются из текста; затем удаляются слова, состоящие из цифр; затем каждое слово заменяется леммой (`MorphVocab`). Функция `getPreparedTexts` возвращает тексты после каждого из трёх видов очистки.

In [8]:
# Совместимость pymorphy2 (его использует natasha) с Python 3.11+: в этих версиях удалена inspect.getargspec
import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

# Подключение элементов библиотеки natasha
from natasha import (
    Segmenter,

    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,

    Doc
)
from natasha import NewsNERTagger
from natasha import MorphVocab

# Модели создаются один раз
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)
morph_vocab = MorphVocab()

In [9]:
# Функция получения подготовленных текстов (выполнение 3-х видов очистки)
def getPreparedTexts(textInitial):
    '''
    Parameters:
    textInitial - Исходный текст

    Returns:
    text_prepared_1 - Подготовленный текст после 1 вида очистки (стоп-слова);
    text_prepared_2 - Подготовленный текст после 2 вида очистки (именованные сущности и цифры);
    text_prepared_3 - Подготовленный текст после 3 вида очистки (лемматизация).
    '''

    textInitial_words_list = re.findall(r'\b\S+\b', textInitial) # Список всех слов в исходном тексте
    textInitial_wordsCount = len(textInitial_words_list)         # Число слов в исходном тексте

    # 1. Удаление стоп-слов
    textInitial_words_list_withoutStopWords = []                 # Список всех слов из текста, кроме стоп-слов
    for i in range(textInitial_wordsCount):
        word = textInitial_words_list[i].lower()                 # Слово (.lower() - с маленькой буквы)
        if not word in stopWords_set:
            textInitial_words_list_withoutStopWords.append(textInitial_words_list[i])
    text_prepared_1 = ' '.join(word for word in textInitial_words_list_withoutStopWords)

    # 2. Удаление именованных сущностей и цифр (библиотека natasha)
    # 2.1 Удаление именованных сущностей
    text = text_prepared_1
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)
    doc.tag_ner(ner_tagger)

    # Поиск именованных сущностей
    namedEntities_list = []               # Именованные сущности в тексте
    namedEntities_count = len(doc.spans)  # Число именованных сущностей в тексте
    for i in range(namedEntities_count):
        namedEntities_list.append(doc.spans[i].text)

    # Удаление именованных сущностей
    text1 = text
    for i in range(namedEntities_count):
        text1 = text1.replace(namedEntities_list[i], '')

    # Удаление лишних двойных и тройных пробелов
    text1 = text1.replace('   ', ' ')
    text1 = text1.replace('  ', ' ')
    text_prepared_2_1 = text1

    # 2.2 Поиск и удаление цифр
    text_prepared_2_1_words_list = re.findall(r'\b\S+\b', text_prepared_2_1)  # Список всех слов
    text_words_list_withoutNumbers = []                                       # Список всех слов из текста, кроме цифр
    for i in range(len(text_prepared_2_1_words_list)):
        word = text_prepared_2_1_words_list[i]
        # Если слово НЕ равно числу, то сохранение в список text_words_list_withoutNumbers
        if word.isdigit() != True:
            text_words_list_withoutNumbers.append(word)
    text_prepared_2 = ' '.join(word for word in text_words_list_withoutNumbers)

    # 3. Лемматизация (лемма - начальная форма слова)
    doc = Doc(text_prepared_2)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    dict_wordAndLemma = {_.text: _.lemma for _ in doc.tokens}   # Словарь [Слово:Лемма]
    text_prepared_3_list = list(dict_wordAndLemma.values())     # Список лемм
    text_prepared_3 = ' '.join(word for word in text_prepared_3_list)

    return text_prepared_1, text_prepared_2, text_prepared_3

In [10]:
# Получение подготовленных текстов (выполнение 3-х видов очистки)
texts_prepared_1 = [] # Подготовленные тексты после 1 вида очистки
texts_prepared_2 = [] # Подготовленные тексты после 2 вида очистки
texts_prepared_3 = [] # Подготовленные тексты после 3 вида очистки
for i in range(articlesCount):
    prepText_1, prepText_2, prepText_3 = getPreparedTexts(textInitial = articlesTexts_list[i])
    texts_prepared_1.append(prepText_1)
    texts_prepared_2.append(prepText_2)
    texts_prepared_3.append(prepText_3)
    print(i, end = ' ')

# Таблица [Рубрика, Текст_статьи, Текст_очищенный_1, Текст_очищенный_2, Текст_очищенный_3]
df_main = pd.DataFrame({'Rubric' : foldersPaths_all_rubrics,
                        'Text' : articlesTexts_list,
                        'Text_prep_1' : texts_prepared_1,
                        'Text_prep_2' : texts_prepared_2,
                        'Text_prep_3' : texts_prepared_3})

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 

In [11]:
# Первая статья после каждого вида очистки
for col in ['Text', 'Text_prep_1', 'Text_prep_2', 'Text_prep_3']:
    print(col + ':')
    print(df_main[col][0][:350])
    print()

# Среднее число слов в статье после каждого вида очистки
word_counts = df_main[['Text', 'Text_prep_1', 'Text_prep_2', 'Text_prep_3']].apply(lambda col: col.str.split().str.len())
print('Среднее число слов в статье:')
print(word_counts.mean().round(1).to_string())

Text:
В компании ожидают что сделка состоится в ближайшее время при условии ее регистрации в местных структурах В Imperial Brands отметили что на Россию и Украину в прошлом году приходилось в общей сложности около 2% чистой выручки и 05% скорректированной операционной прибыли компании Imperial Brands 9 марта объявила о приостановке производства прекращен

Text_prep_1:
компании ожидают сделка состоится ближайшее регистрации местных структурах Imperial Brands отметили Россию Украину прошлом приходилось общей сложности 2 чистой выручки 05 скорректированной операционной прибыли компании Imperial Brands 9 марта объявила приостановке производства прекращении маркетинговой деятельности территории компании объяснили реш

Text_prep_2:
компании ожидают сделка состоится ближайшее регистрации местных структурах отметили прошлом приходилось общей сложности чистой выручки скорректированной операционной прибыли компании марта объявила приостановке производства прекращении маркетинговой деятельности т

**Как читать результат.** Каждый вид очистки применяется к результату предыдущего, поэтому среднее число слов в таблице выше показывает вклад каждого шага: удаление стоп-слов убирает служебные слова, удаление сущностей и чисел убирает названия компаний, имена, даты и суммы. На третьем шаге слова заменяются леммами (формы «компании», «компания», «компанию» превращаются в «компания»), поэтому одно слово в разных падежах и числах считается одним признаком.

**Особенность кода методички на этапе лемматизации.** Список лемм строится из словаря `{слово: лемма}`, то есть одинаковые словоформы в тексте схлопываются в одну запись. Поэтому в `Text_prep_3` каждая словоформа встречается один раз, а повторы одного и того же слова в статье теряются (разные формы одного слова, например «компании» и «компания», остаются отдельными записями, но получают одну лемму и дают повтор леммы). Это влияет на TF: частота слова внутри статьи отражает в основном число его разных форм, а не число упоминаний. Метод от этого работает, но вес слов в TF-IDF отличается от «классического».

### 4.3. Сохранение подготовленных текстов

In [12]:
# Сохранение подготовленных текстов (папки создавались на этапе 2)
for i in range(articlesCount):
    with open(foldersPaths_preparedData_all[i], 'w', encoding = 'utf8') as text_file:
        text_file.write(texts_prepared_3[i])

print('Сохранено файлов:', len(foldersPaths_preparedData_all), '| папка:', folderPath_preparedData)

Сохранено файлов: 45 | папка: c:\Users\asus\OneDrive\Документы\2coursepraktika\ТехнБольшихДанных\inwork\lab_2_5\DATA\Data_prepared


## 5. Модель KNN

### 5.1. Импорт подготовленных текстов

In [13]:
# Список текстов подготовленных
articlesTexts_prepared_list = []
for i in range(articlesCount):
    with open(foldersPaths_preparedData_all[i], 'r', encoding = 'utf8') as text_file:
        contents_list = text_file.readlines()                    # Текст в файле (построчно)
        articlesTexts_prepared_list.append(contents_list[0])     # Сохранение текста в список

# Список имен файлов с текстами
articlesTexts_prepared_filesNames = [os.path.basename(path) for path in foldersPaths_preparedData_all]

# Номера рубрик
rubricsNumbers_list = []        # Номера рубрик для всех статей (n = числу статей)
for i in range(articlesCount):
    rubricName = df_rubricsAndArticlesPaths_all.iloc[i, 0]
    # Поиск номера рубрики
    for j in range(rubricsCount):
        if rubricName == foldersNames_list[j]:
            rubricNumber = j
    rubricsNumbers_list.append(rubricNumber)

# Таблица [Номер_рубрики, Рубрика, Файл_текста, Текст_подготовленный]
df_preparedTexts = pd.DataFrame({'RubricNumber' : rubricsNumbers_list,
                                 'Rubric' : foldersPaths_all_rubrics,
                                 'FileName' : articlesTexts_prepared_filesNames,
                                 'TextPrep' : articlesTexts_prepared_list})
df_preparedTexts.head()

,RubricNumber,Rubric,FileName,TextPrep
0,0,Рубрика бизнес,1.txt,компания ожидать сделка состояться близкий рег...
1,0,Рубрика бизнес,10.txt,оператор каршеринг подходить конец лизинговый ...
2,0,Рубрика бизнес,11.txt,японский корпорация приостанавливать прием зак...
3,0,Рубрика бизнес,12.txt,американский бизнесмен готовый инвестировать 1...
4,0,Рубрика бизнес,13.txt,выделить млрд поддержка коммерческий атомный э...


### 5.2. Представление текстов числовыми векторами (TF-IDF)

Параметры `TfidfVectorizer` те же, что в методичке: `sublinear_tf=True` (вместо частоты $tf$ берётся $1 + \ln tf$, чтобы повторы слова не росли линейно), `max_df=0.5` (слова, которые встречаются больше чем в половине статей, то есть не менее чем в 23 из 45, отбрасываются как неинформативные), `stop_words=None` (стоп-слова уже удалены). Векторизатор по умолчанию нормирует каждый вектор до единичной длины (норма L2).

In [14]:
# Представление текстов в виде числовых векторов
docs = articlesTexts_prepared_list #    Документы (тексты)
vectorizer = TfidfVectorizer(sublinear_tf = True, max_df = 0.5, stop_words = None)
res = vectorizer.fit_transform(docs)
res_words = vectorizer.get_feature_names_out()   # Список слов из всех текстов
res_mx = res.toarray()                           # Тексты в виде числовых векторов. Матрица [Число_документов х Число_уникальных_слов_всего]
res_idf = vectorizer.idf_

numericVectors_mx = res_mx

# Сохранение числовых векторов в таблицу df_preparedTexts
df_preparedTexts['NumericVector'] = list(numericVectors_mx)

print('Размер матрицы [документы x слова]:', numericVectors_mx.shape)
print('Длины векторов (норма L2), мин/макс:', np.linalg.norm(numericVectors_mx, axis = 1).min().round(6), np.linalg.norm(numericVectors_mx, axis = 1).max().round(6))

Размер матрицы [документы x слова]: (45, 1430)
Длины векторов (норма L2), мин/макс: 1.0 1.0


In [15]:
# Слова с наибольшим средним весом TF-IDF в каждой рубрике
tfidf_df = pd.DataFrame(numericVectors_mx, columns = res_words)
for rubric in foldersNames_list:
    mean_weights = tfidf_df[df_preparedTexts['Rubric'] == rubric].mean().sort_values(ascending = False)
    print(rubric + ':', ', '.join(mean_weights.index[:8]))

Рубрика бизнес: компания, сообщить, ситуация, источник, заявление, российский, сообщать, пресс
Рубрика культура: проект, выставка, апрель, посвятить, представить, конкурс, экспозиция, культурный
Рубрика спорт: чемпионат, спорт, матч, команда, турнир, победа, спортсмен, чемпион


Слова с наибольшим весом показывают, по каким признакам KNN будет различать рубрики: если в списках видны слова по теме рубрики (для спорта — матч, команда и т.п.), векторное представление работает так, как задумано в методичке. Реальные значения: для спорта (чемпионат, спорт, матч, команда, турнир) и культуры (проект, выставка, экспозиция, конкурс) в списке стоят слова по теме рубрики. Для бизнеса вместе со словом «компания» в списке стоят общие слова новостного стиля («сообщить», «источник», «заявление», «пресс»), а не только экономические термины. Это не ошибка расчёта: короткие деловые заметки состоят в основном из пересказа заявлений, и KNN отличает их от культуры и спорта именно по таким словам.

Число столбцов матрицы равно числу уникальных лемм во всех 45 статьях, поэтому оно зависит от списка стоп-слов и версии `natasha`; точное значение не фиксируется методичкой. Реальное значение: 1430 уникальных лемм (матрица 45 × 1430).

Векторы нормированы по длине, поэтому евклидово расстояние (формула (2.5) методички, метрика `KNeighborsClassifier` по умолчанию) однозначно связано с косинусным сходством текстов: $d^2 = 2 - 2\cos\theta$.

### 5.3. Обучение модели KNN и оценка качества

Разделение на обучающую (75 %) и тестовую (25 % = 12 текстов) выборки; число соседей $k = 4$ и простое невзвешенное голосование, как в методичке.

In [16]:
# Обучение модели KNN
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

texts_dataset_data = numericVectors_mx                # Тексты в виде числовых векторов (X)
texts_dataset_target = np.array(rubricsNumbers_list)  # Метки текстов (номер рубрики для текста) (Y)

# random_state фиксирует разбиение; idx_train, idx_test - номера статей в таблице df_preparedTexts
X_train, X_test, Y_train, Y_test, idx_train, idx_test = train_test_split(texts_dataset_data, texts_dataset_target,
                                                                         np.arange(articlesCount), random_state = 42)

model_KNN = KNeighborsClassifier(n_neighbors = 4) # Число соседей
model_KNN.fit(X_train, Y_train)

predicted = model_KNN.predict(X_test)

print(metrics.classification_report(Y_test, predicted))
print(metrics.confusion_matrix(Y_test, predicted))
print()
print('Номера рубрик:', dict(enumerate(foldersNames_list)))
print('Тестовых текстов по рубрикам (support):', {int(k): int(v) for k, v in zip(*np.unique(Y_test, return_counts = True))})

              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       1.00      1.00      1.00         3
           2       1.00      0.75      0.86         4

    accuracy                           0.92        12
   macro avg       0.94      0.92      0.92        12
weighted avg       0.93      0.92      0.91        12

[[5 0 0]
 [0 3 0]
 [1 0 3]]

Номера рубрик: {0: 'Рубрика бизнес', 1: 'Рубрика культура', 2: 'Рубрика спорт'}
Тестовых текстов по рубрикам (support): {0: 5, 1: 3, 2: 4}


**Как сравнивать с рис. 2.45.** На рис. 2.45 тестовая выборка состоит из 12 текстов (5 + 4 + 3 по рубрикам), accuracy $= 0{,}92$ (одна ошибка: один текст рубрики 2 «спорт» отнесён к рубрике 1 «культура»). В методичке `train_test_split` вызывается без `random_state`, поэтому состав тестовой выборки при каждом запуске случайный и точное совпадение с рисунком не гарантируется. Метрики на 12 текстах дискретны: каждый неверный текст снижает accuracy на $1/12 \approx 0{,}083$ (11 верных из 12 — $0{,}92$; 12 из 12 — $1{,}00$), а recall отдельной рубрики меняется шагом $1/n$, где $n$ — число тестовых текстов рубрики (здесь от 3 до 5). Поэтому ниже добавлена устойчивая оценка по кросс-валидации.

**Реальные значения против рис. 2.45.** Тестовая выборка — 12 текстов, accuracy $= 0{,}92$ (11 верных из 12), то есть итоговая accuracy совпала с методичкой. Отличаются детали разбиения и ошибки:

| | Методичка (рис. 2.45) | Реальные значения |
|---|---|---|
| Состав тестовой выборки (бизнес + культура + спорт) | 5 + 4 + 3 по рубрикам | 5 + 3 + 4 |
| Единственная ошибка | текст рубрики «спорт» отнесён к «культуре» | текст рубрики «спорт» отнесён к «бизнесу» (строка спорта в матрице ошибок `[1 0 3]`) |
| Метрики, затронутые ошибкой | — | recall спорта $0{,}75$, precision бизнеса $0{,}83$; культура определена без ошибок (precision и recall $1{,}00$) |

Оба результата — одна ошибка на 12 текстов, поэтому расхождение в составе и в типе ошибки объясняется только случайным разбиением и не означает, что модель или код работают иначе, чем в методичке.

In [17]:
# Дополнительная проверка (в методичке отсутствует): 5-кратная стратифицированная кросс-валидация.
# TF-IDF обучается внутри каждого блока, поэтому тестовые тексты не влияют на словарь и IDF.
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
rows = []
for name, texts in [('1. после удаления стоп-слов', texts_prepared_1),
                    ('2. + удаление сущностей и чисел', texts_prepared_2),
                    ('3. + лемматизация (итоговый вариант)', articlesTexts_prepared_list)]:
    pipe = make_pipeline(TfidfVectorizer(sublinear_tf = True, max_df = 0.5, stop_words = None),
                         KNeighborsClassifier(n_neighbors = 4))
    scores = cross_val_score(pipe, texts, texts_dataset_target, cv = cv, scoring = 'accuracy')
    rows.append({'вид очистки': name, 'accuracy (среднее)': scores.mean().round(3), 'min': scores.min().round(3), 'max': scores.max().round(3)})
print(pd.DataFrame(rows).to_string(index = False))

                         вид очистки  accuracy (среднее)   min  max
         1. после удаления стоп-слов               0.867 0.778  1.0
     2. + удаление сущностей и чисел               0.889 0.778  1.0
3. + лемматизация (итоговый вариант)               0.867 0.667  1.0


**Как читать таблицу.** В кросс-валидации каждый из 45 текстов ровно один раз оказывается в тестовой части (по 9 текстов в блоке), поэтому среднее по пяти блокам устойчивее, чем одно разбиение на 12 текстов. Разброс `min`–`max` показывает, насколько результат зависит от состава блока. Сравнение трёх строк показывает вклад каждого вида очистки в качество; методика не требует, чтобы лемматизация повышала accuracy, она в первую очередь уменьшает размерность признакового пространства.

**Реальные значения.** Средняя accuracy по пяти блокам: $0{,}867$ после удаления стоп-слов ($39$ верных из $45$), $0{,}889$ после удаления сущностей и чисел ($40$ из $45$), $0{,}867$ для итогового варианта с леммами ($39$ из $45$). Строки различаются на один текст из 45 ($\approx 0{,}022$), поэтому ни один вид очистки нельзя считать заметно лучше другого: на 45 статьях разница в один текст находится в пределах случайного разброса, а `min`–`max` от $0{,}667$ до $1{,}0$ показывает, что результат отдельного блока из 9 текстов колеблется сильнее. Кросс-валидация ($0{,}87$) немного ниже, чем accuracy одного разбиения ($0{,}92$). Одна из причин: в разбиении 5.3, как в методичке, TF-IDF обучается на всех 45 текстах до разбиения, а в кросс-валидации словарь и IDF строятся только по обучающей части блока, поэтому её оценка честнее.

### 5.4. Применение модели к тексту

In [18]:
# Определение текста, который соответствует первому вектору из тестовой выборки
idx_text = idx_test[0]                          # Индекс нужного текста в таблице
text_text = df_preparedTexts['TextPrep'][idx_text]

# Применение модели для первого текста тестовой выборки
predicted_text1 = model_KNN.predict(X_test[:1])
print('\nПервый текст из тестовой выборки следующий:')
print(text_text)

print('\nПервый текст из тестовой выборки относится к следующей рубрике')
print('Номер рубрики: ', predicted_text1[0])
print('Название рубрики: ', foldersNames_list[predicted_text1[0]])
print('Реальная рубрика: ', df_preparedTexts['Rubric'][idx_text])


Первый текст из тестовой выборки следующий:
одержать победа счет 3 : 2 матч чемпионат сообщать корреспондент лентыр российский хавбек помочь команда сравнять счет начало встреча 12-й минута отметиться результативный передача позволить забить гол монегаск забить гол обыграть соперник провести матч поле заменить 78-й игра вступить нынешний розыгрыш матч россиянин два забитый мяч четыре результативный передача перейти продать игрок евро

Первый текст из тестовой выборки относится к следующей рубрике
Номер рубрики:  2
Название рубрики:  Рубрика спорт
Реальная рубрика:  Рубрика спорт


Рис. 2.46–2.48 методички показывают по одному тестированию для каждой рубрики. Ниже то же самое выполнено для первого текста тестовой выборки каждой рубрики; выводится реальная рубрика, ответ модели и доли голосов четырёх соседей (при $k=4$ возможна ничья 2:2, в этом случае `scikit-learn` выбирает рубрику с меньшим номером).

In [19]:
# Функция классификации произвольного «сырого» текста: очистка -> стоп-слова/сущности/цифры/леммы -> TF-IDF -> KNN
import io

def classify_text(raw_text, model = model_KNN, vectorizer = vectorizer):
    text = cleanLines(io.StringIO(raw_text).readlines())
    prepared = getPreparedTexts(text)[2]
    vec = vectorizer.transform([prepared])
    probs = model.predict_proba(vec)[0]
    return foldersNames_list[int(model.predict(vec)[0])], prepared, probs, vec.nnz


for number in range(rubricsCount):
    tests_in_rubric = [i for i in idx_test if rubricsNumbers_list[i] == number]
    if not tests_in_rubric:
        print(foldersNames_list[number] + ': в тестовой выборке нет текстов этой рубрики\n')
        continue
    i = tests_in_rubric[0]
    with open(foldersPaths_all[i], 'r', encoding = 'utf8') as f:
        raw = f.read()
    answer, prepared, probs, nnz = classify_text(raw)
    print('Реальная рубрика:', foldersNames_list[number], '| ответ модели:', answer,
          '| доли голосов соседей:', {r: float(p) for r, p in zip(foldersNames_list, probs.round(2))})
    print(prepared[:200], '\n')

Реальная рубрика: Рубрика бизнес | ответ модели: Рубрика бизнес | доли голосов соседей: {'Рубрика бизнес': 1.0, 'Рубрика культура': 0.0, 'Рубрика спорт': 0.0}
выделить млрд поддержка коммерческий атомный электростанция находиться угроза закрытие финансовый трудность сообщать газета c о ссылка заявление министр энергетика производить половина безуглеродный э 

Реальная рубрика: Рубрика культура | ответ модели: Рубрика культура | доли голосов соседей: {'Рубрика бизнес': 0.0, 'Рубрика культура': 1.0, 'Рубрика спорт': 0.0}
май проект проводить прием заявка грантовый конкурс цель поощрять создание развитие инклюзивный проект ребенок особенность ментальный развитие популяризация профессиональный сообщество широкий публика 

Реальная рубрика: Рубрика спорт | ответ модели: Рубрика спорт | доли голосов соседей: {'Рубрика бизнес': 0.0, 'Рубрика культура': 0.0, 'Рубрика спорт': 1.0}
одержать победа счет 3 : 2 матч чемпионат сообщать корреспондент лентыр российский хавбек помочь команда сравнять с

In [20]:
# Проверка на новых текстах, не входящих в датасет (короткие тексты, написанные для проверки)
new_texts = {
    'бизнес': 'Компания объявила о снижении выручки и сообщила, что акционеры одобрили сделку по продаже бизнеса. Банк повысил ставку по кредитам, а инвесторы ожидают роста цен на нефть и падения курса рубля.',
    'культура': 'В музее открылась выставка современных художников, на которой представлены картины и скульптуры. Режиссёр представил новый спектакль, а театр объявил о премьере и фестивале искусства.',
    'спорт': 'Футбольная команда выиграла матч у соперника со счётом два один и вышла в финал чемпионата. Тренер похвалил игроков, а нападающий забил победный гол на последней минуте.',
}
for true_name, raw in new_texts.items():
    answer, prepared, probs, nnz = classify_text(raw)
    print('Ожидается: рубрика', true_name, '| ответ модели:', answer, '| доли голосов:', {r: float(p) for r, p in zip(foldersNames_list, probs.round(2))})

Ожидается: рубрика бизнес | ответ модели: Рубрика бизнес | доли голосов: {'Рубрика бизнес': 1.0, 'Рубрика культура': 0.0, 'Рубрика спорт': 0.0}
Ожидается: рубрика культура | ответ модели: Рубрика культура | доли голосов: {'Рубрика бизнес': 0.0, 'Рубрика культура': 1.0, 'Рубрика спорт': 0.0}
Ожидается: рубрика спорт | ответ модели: Рубрика спорт | доли голосов: {'Рубрика бизнес': 0.0, 'Рубрика культура': 0.0, 'Рубрика спорт': 1.0}


Тексты в последней ячейке написаны для проверки и в датасет не входят; они короткие (около 30 слов, в датасете — 58–143), поэтому ответ модели менее надёжен, чем на статьях датасета. Доли голосов показывают уверенность: 1,00 — все четыре соседа из одной рубрики, 0,50 — ничья.

## Самостоятельная работа

### 1. Классификатор для трёх других категорий

Весь конвейер (чтение, очистка, `natasha`, TF-IDF, KNN, метрики) собран в функцию `build_text_classifier`, которая принимает папку с тремя рубриками. Требуется структура, как у `Data`: `Data_selfwork/<название рубрики>/*.txt` (кодировка UTF-8, не менее 15 статей в рубрике, не менее 50–100 слов в статье), например «кулинария», «компьютерные игры», «high tech». Папка с данными, в отличие от первой части, в методичке не приложена. Для этого пункта тексты взяты из корпуса новостей Lenta.ru (столбцы `topic` — рубрика, `text` — текст): отдельным скриптом из корпуса случайно отобраны короткие статьи (по умолчанию 50–150 слов, как в `Data.zip`) трёх рубрик, каждая статья сохранена в отдельный `.txt`. Названия рубрик, число статей и их длина определяются по содержимому папки `Data_selfwork` и выводятся ячейками ниже. Папка должна лежать в `DATA` (там же, где `Data`).

План пункта: 1) прочитать статьи и выполнить три вида очистки, как в основной части; 2) построить TF-IDF и KNN (`k = 4`), вывести метрики на тестовой выборке; 3) описать данные и слова с наибольшим весом; 4) проверить устойчивость кросс-валидацией для трёх видов очистки; 5) разобрать ответы модели на тестовых статьях и ошибки.

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

def load_corpus(data_dir):
    '''Читает статьи из data_dir (внутри - папки рубрик с файлами .txt) и выполняет три вида очистки (как в разделе 4).'''
    rubrics = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
    rows = []
    for number, rubric in enumerate(rubrics):
        for path in getFilesPathsInRubricFolder(os.path.join(data_dir, rubric)):
            text = readTextFromFile(path)
            prep_1, prep_2, prep_3 = getPreparedTexts(text)
            rows.append({'RubricNumber' : number, 'Rubric' : rubric, 'File' : os.path.basename(path), 'Text' : text,
                         'Text_prep_1' : prep_1, 'Text_prep_2' : prep_2, 'Text_prep_3' : prep_3})
    return rubrics, pd.DataFrame(rows)


def build_text_classifier(data_dir, n_neighbors = 4, random_state = 42):
    '''Строит классификатор текстов по папке data_dir: очистка -> TF-IDF -> KNN -> метрики на тестовой выборке.'''
    rubrics, df = load_corpus(data_dir)
    vec = TfidfVectorizer(sublinear_tf = True, max_df = 0.5, stop_words = None)
    X = vec.fit_transform(df['Text_prep_3']).toarray()
    y = df['RubricNumber'].values

    X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(X, y, np.arange(len(df)), random_state = random_state)
    model = KNeighborsClassifier(n_neighbors = n_neighbors).fit(X_tr, y_tr)
    pred = model.predict(X_te)

    labels = list(range(len(rubrics)))
    print('Рубрики:', dict(enumerate(rubrics)), '| статей:', len(df), '| слов в словаре:', X.shape[1])
    print('Тестовых текстов по рубрикам (support):', {rubrics[k] : int((y_te == k).sum()) for k in labels})
    print(classification_report(y_te, pred, labels = labels, target_names = rubrics, zero_division = 0))
    print(confusion_matrix(y_te, pred, labels = labels))
    return {'model' : model, 'vectorizer' : vec, 'rubrics' : rubrics, 'corpus' : df, 'X' : X, 'y' : y, 'idx_test' : idx_te}


selfwork_dir = os.path.join(cwd_DATA, 'Data_selfwork')    # Data_selfwork/<рубрика 1>, <рубрика 2>, <рубрика 3>
if os.path.isdir(selfwork_dir):
    classifier_selfwork = build_text_classifier(selfwork_dir)
else:
    print('Папка Data_selfwork не найдена: для этого пункта нужны статьи трёх других рубрик (см. описание выше).')

Рубрики: {0: 'Рубрика наука и техника', 1: 'Рубрика путешествия', 2: 'Рубрика силовые структуры'} | статей: 60 | слов в словаре: 1971
Тестовых текстов по рубрикам (support): {'Рубрика наука и техника': 5, 'Рубрика путешествия': 3, 'Рубрика силовые структуры': 7}
                           precision    recall  f1-score   support

  Рубрика наука и техника       1.00      0.60      0.75         5
      Рубрика путешествия       0.50      1.00      0.67         3
Рубрика силовые структуры       1.00      0.86      0.92         7

                 accuracy                           0.80        15
                macro avg       0.83      0.82      0.78        15
             weighted avg       0.90      0.80      0.81        15

[[3 2 0]
 [0 3 0]
 [0 1 6]]


**Как читать вывод.** Номера рубрик 0, 1, 2 соответствуют названиям папок по алфавиту (так задаёт `sorted`). Тестовая выборка — 25 % статей, то есть при 60 статьях 15 текстов, а при 45 статьях 12; поэтому accuracy меняется шагом $1/n_{test}$, а recall каждой рубрики — шагом $1/n$ по числу её тестовых текстов. Эталонных чисел для этого пункта в методичке нет (она предлагает взять любые три категории), так что сверять результат не с чем: ориентир — случайное угадывание, которое для трёх равных по размеру рубрик даёт accuracy около $1/3 \approx 0{,}33$. Значения, заметно превосходящие этот уровень, показывают, что TF-IDF и KNN различают рубрики так же, как в основной части.

In [22]:
# Описание собранных данных (выводится только если папка Data_selfwork найдена)
if 'classifier_selfwork' in globals():
    df_sw = classifier_selfwork['corpus'].copy()
    df_sw['Words_raw'] = df_sw['Text'].str.split().str.len()

    summary = df_sw.groupby('Rubric')['Words_raw'].agg(['count', 'min', 'mean', 'max']).round(1)
    summary.columns = ['статей', 'слов (мин)', 'слов (среднее)', 'слов (макс)']
    print(summary.to_string())
    print()
    print('Среднее число слов в статье после каждого вида очистки:')
    print(df_sw[['Text', 'Text_prep_1', 'Text_prep_2', 'Text_prep_3']].apply(lambda col: col.str.split().str.len()).mean().round(1).to_string())
else:
    print('Сначала нужна папка Data_selfwork и выполненная ячейка выше.')

                           статей  слов (мин)  слов (среднее)  слов (макс)
Rubric                                                                    
Рубрика наука и техника        20          51           114.1          147
Рубрика путешествия            20          73           120.6          146
Рубрика силовые структуры      20          77           123.2          148

Среднее число слов в статье после каждого вида очистки:
Text           119.3
Text_prep_1     80.2
Text_prep_2     65.1
Text_prep_3     60.7


In [23]:
# Слова с наибольшим средним весом TF-IDF в каждой рубрике (как в п. 5.2)
if 'classifier_selfwork' in globals():
    words_sw = classifier_selfwork['vectorizer'].get_feature_names_out()
    tfidf_sw = pd.DataFrame(classifier_selfwork['X'], columns = words_sw)
    for number, rubric in enumerate(classifier_selfwork['rubrics']):
        mean_weights = tfidf_sw[df_sw['RubricNumber'] == number].mean().sort_values(ascending = False)
        print(rubric + ':', ', '.join(mean_weights.index[:8]))
else:
    print('Сначала нужна папка Data_selfwork и выполненная ячейка выше.')

Рубрика наука и техника: ракета, носитель, спутник, диск, аппарат, новый, система, представитель
Рубрика путешествия: пассажир, турист, аэропорт, самолет, авиакомпания, воздушный, рейс, путешественник
Рубрика силовые структуры: военный, возбудить, преступление, задержать, уголовный, сообщить, управление, орган


**Как читать две таблицы выше.** Первая показывает размер и длину статей: все статьи должны быть короткими (не короче 50 слов, это условие отбора), а среднее число слов должно уменьшаться от шага к шагу, как в основной части (стоп-слова, затем сущности и числа, затем повторы словоформ при лемматизации). Вторая показывает, по каким словам KNN отличает рубрики: если в строке рубрики стоят слова по её теме, векторное представление работает так, как задумано. Слова общего новостного стиля («сообщить», «заявить», «источник») в списке той или иной рубрики не ошибка: так же было у рубрики «бизнес» в основной части, где новости построены на пересказе заявлений.

In [24]:
# Устойчивая оценка: 5-кратная стратифицированная кросс-валидация для трёх видов очистки (как в п. 5.3)
if 'classifier_selfwork' in globals():
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    y_sw = classifier_selfwork['y']
    n_splits_sw = min(5, int(np.bincount(y_sw).min()))     # в каждой рубрике должно быть не меньше статей, чем блоков
    cv_sw = StratifiedKFold(n_splits = n_splits_sw, shuffle = True, random_state = 42)
    rows_sw = []
    for name, column in [('1. после удаления стоп-слов', 'Text_prep_1'),
                         ('2. + удаление сущностей и чисел', 'Text_prep_2'),
                         ('3. + лемматизация (итоговый вариант)', 'Text_prep_3')]:
        pipe = make_pipeline(TfidfVectorizer(sublinear_tf = True, max_df = 0.5, stop_words = None),
                             KNeighborsClassifier(n_neighbors = 4))
        scores = cross_val_score(pipe, df_sw[column], y_sw, cv = cv_sw, scoring = 'accuracy')
        rows_sw.append({'вид очистки' : name, 'accuracy (среднее)' : scores.mean().round(3), 'min' : scores.min().round(3), 'max' : scores.max().round(3)})
    print('Число блоков:', n_splits_sw, '| статей в блоке:', len(y_sw) // n_splits_sw)
    print(pd.DataFrame(rows_sw).to_string(index = False))
else:
    print('Сначала нужна папка Data_selfwork и выполненная ячейка выше.')

Число блоков: 5 | статей в блоке: 12
                         вид очистки  accuracy (среднее)   min   max
         1. после удаления стоп-слов               0.733 0.583 0.917
     2. + удаление сущностей и чисел               0.733 0.500 0.917
3. + лемматизация (итоговый вариант)               0.833 0.667 1.000


**Как читать таблицу.** Это та же проверка, что в п. 5.3 основной части: каждая статья один раз попадает в тестовую часть, TF-IDF обучается внутри блока, поэтому оценка честнее, чем accuracy одного разбиения (в одном разбиении, как в методичке, словарь и IDF считаются по всем статьям до деления). Строки таблицы различаются обычно на одну-две статьи из всего набора ($1/n \approx 0{,}02$), поэтому небольшая разница между видами очистки находится в пределах случайного разброса; лемматизация в первую очередь уменьшает размер словаря и не обязана повышать accuracy.

In [25]:
# Ответы модели на тестовых статьях: по одной статье каждой рубрики и все ошибки
if 'classifier_selfwork' in globals():
    model_sw = classifier_selfwork['model']
    vec_sw = classifier_selfwork['vectorizer']
    rubrics_sw = classifier_selfwork['rubrics']
    idx_test_sw = classifier_selfwork['idx_test']

    for number, rubric in enumerate(rubrics_sw):
        in_rubric = [i for i in idx_test_sw if df_sw['RubricNumber'][i] == number]
        if not in_rubric:
            print(rubric + ': в тестовой выборке нет текстов этой рубрики\n')
            continue
        i = in_rubric[0]
        vec_row = vec_sw.transform([df_sw['Text_prep_3'][i]])
        probs = model_sw.predict_proba(vec_row)[0]
        print('Реальная рубрика:', rubric, '| ответ модели:', rubrics_sw[int(model_sw.predict(vec_row)[0])],
              '| доли голосов соседей:', {r : float(p) for r, p in zip(rubrics_sw, probs.round(2))})
        print(df_sw['Text'][i][:200], '\n')

    predicted_sw = model_sw.predict(classifier_selfwork['X'][idx_test_sw])
    wrong = [(i, p) for i, p in zip(idx_test_sw, predicted_sw) if p != df_sw['RubricNumber'][i]]
    print('Ошибок на тестовой выборке:', len(wrong), 'из', len(idx_test_sw))
    for i, p in wrong:
        print('-', df_sw['File'][i], '| реальная рубрика:', df_sw['Rubric'][i], '| ответ модели:', rubrics_sw[p])
        print('  ', df_sw['Text'][i][:200])
else:
    print('Сначала нужна папка Data_selfwork и выполненная ячейка выше.')

Реальная рубрика: Рубрика наука и техника | ответ модели: Рубрика наука и техника | доли голосов соседей: {'Рубрика наука и техника': 0.75, 'Рубрика путешествия': 0.25, 'Рубрика силовые структуры': 0.0}
Компания Blizzard Entertainment объявила что одновременно с обычной версией Burning Crusade дополнения к популярной многопользовательской игре World of Warcraft выпустит еще и коллекционное издание В  

Реальная рубрика: Рубрика путешествия | ответ модели: Рубрика путешествия | доли голосов соседей: {'Рубрика наука и техника': 0.0, 'Рубрика путешествия': 1.0, 'Рубрика силовые структуры': 0.0}
Пассажира рейса Копенгаген Эдинбург британского лоукостера EasyJet после приземления задержала вызванная на борт полиция мужчина не подчинялся членам экипажа и оскорблял окружающих сообщает Daily Mail 

Реальная рубрика: Рубрика силовые структуры | ответ модели: Рубрика силовые структуры | доли голосов соседей: {'Рубрика наука и техника': 0.0, 'Рубрика путешествия': 0.25, 'Рубрика силовые структуры

**Как читать вывод.** Для каждой рубрики показана первая статья тестовой выборки: реальная рубрика, ответ модели и доли голосов четырёх соседей (1,00 — все соседи из одной рубрики, 0,50 — ничья 2:2, при ничьей `scikit-learn` выбирает рубрику с меньшим номером). В списке ошибок для каждой ошибочной статьи указан файл, реальная рубрика и ответ модели; такие статьи обычно лежат на стыке рубрик (например, заметка о технологиях в военной или транспортной сфере) или состоят в основном из слов общего новостного стиля. Отдельные ошибки на 12–15 тестовых текстах не говорят о неисправности кода: каждая ошибка снижает accuracy на $1/n_{test}$, как и в основной части.

In [26]:
# Общая модель для бота: рубрики из Data + рубрики из Data_selfwork, обучение на всех статьях
if 'classifier_selfwork' in globals():
    import joblib

    rubrics_all = foldersNames_list + classifier_selfwork['rubrics']                   # 3 старые + 3 новые рубрики
    texts_all = list(articlesTexts_prepared_list) + list(classifier_selfwork['corpus']['Text_prep_3'])
    labels_all = np.concatenate([texts_dataset_target, classifier_selfwork['y'] + rubricsCount])   # номера новых рубрик сдвинуты

    vectorizer_all = TfidfVectorizer(sublinear_tf = True, max_df = 0.5, stop_words = None)
    model_all = KNeighborsClassifier(n_neighbors = 4).fit(vectorizer_all.fit_transform(texts_all).toarray(), labels_all)
    joblib.dump({'vectorizer': vectorizer_all, 'model': model_all, 'rubrics': rubrics_all}, 'knn_text_classifier.joblib')
    print('Общая модель для бота сохранена в knn_text_classifier.joblib | рубрик:', len(rubrics_all), '| статей:', len(texts_all))
    print(rubrics_all)
else:
    print('Папка Data_selfwork не обработана: для бота будет сохранена модель только по рубрикам из Data.')

Общая модель для бота сохранена в knn_text_classifier.joblib | рубрик: 6 | статей: 105
['Рубрика бизнес', 'Рубрика культура', 'Рубрика спорт', 'Рубрика наука и техника', 'Рубрика путешествия', 'Рубрика силовые структуры']


### 2. Телеграмм-бот


In [27]:
import importlib
import telegram_bot
importlib.reload(telegram_bot)

# Статьи для сверки, подготовленные тексты ноутбука и модель, с которой должен совпасть бот
check_paths = list(foldersPaths_all)
check_prepared = list(texts_prepared_3)
ref_vectorizer, ref_model, ref_rubrics = vectorizer, model_all, foldersNames_list
if 'model_all' in globals():
    corpus_sw = classifier_selfwork['corpus']
    check_paths += [os.path.join(selfwork_dir, rubric, file) for rubric, file in zip(corpus_sw['Rubric'], corpus_sw['File'])]
    check_prepared += list(corpus_sw['Text_prep_3'])
    ref_vectorizer, ref_model, ref_rubrics = vectorizer_all, model_all, rubrics_all

same_prepared, same_answer = 0, 0
for path, prepared in zip(check_paths, check_prepared):
    with open(path, 'r', encoding = 'utf8') as f:
        raw = f.read()
    same_prepared += int(telegram_bot.prepare_text(raw) == prepared)
    answer_bot, _ = telegram_bot.predict_rubric(raw)
    vec = ref_vectorizer.transform([prepared])
    same_answer += int(answer_bot == ref_rubrics[int(ref_model.predict(vec)[0])])

print('Рубрики бота:', telegram_bot.rubrics)
print('Совпадение подготовленных текстов бота и ноутбука:', same_prepared, 'из', len(check_paths))
print('Совпадение ответов бота и модели из ноутбука    :', same_answer, 'из', len(check_paths))
example_name, example_probs = telegram_bot.predict_rubric(new_texts['спорт'])
print('Пример ответа бота:', example_name, {r: float(p) for r, p in example_probs.items()})

модели загружены 3.3 с
модели загружены 0.7 с
Рубрики бота: ['Рубрика бизнес', 'Рубрика культура', 'Рубрика спорт', 'Рубрика наука и техника', 'Рубрика путешествия', 'Рубрика силовые структуры']
Совпадение подготовленных текстов бота и ноутбука: 105 из 105
Совпадение ответов бота и модели из ноутбука    : 105 из 105
Пример ответа бота: Рубрика спорт {'Рубрика бизнес': 0.0, 'Рубрика культура': 0.0, 'Рубрика спорт': 1.0, 'Рубрика наука и техника': 0.0, 'Рубрика путешествия': 0.0, 'Рубрика силовые структуры': 0.0}


Ограничения. Ответ KNN определяется ближайшими статьями из всех обучающих (45 из `Data` и статьи из `Data_selfwork`); на текстах другой тематики, чем рубрики модели, бот всё равно выберет одну из них (модели нет класса «другое»). Очень короткие сообщения дают малое число слов в словаре и ненадёжный ответ: бот просит не менее 5 слов, а надёжной считается статья в 50 слов и больше (как в датасете).

## Итоги сверки с методичкой

| Что | В методичке | Реальные значения / комментарий |
|---|---|---|
| Датасет | 45 статей, 15 в рубрике | 3 рубрики по 15 статей, совпадает |
| Номера рубрик | 0 — бизнес, 1 — культура, 2 — спорт (рис. 2.46–2.48) | совпадает; порядок гарантирован сортировкой `sorted(os.listdir())` |
| Три вида очистки | стоп-слова, сущности и цифры, леммы | реализованы по методичке; модели `natasha` создаются один раз |
| Метрики модели (рис. 2.45) | accuracy 0,92 на 12 тестовых текстах | accuracy $0{,}92$ совпала с методичкой; состав выборки 5 + 3 + 4 (в методичке 5 + 4 + 3), единственная ошибка «спорт» → «бизнес» (в методичке «спорт» → «культура»); разбиение зафиксировано `random_state=42`, дополнительно приведена кросс-валидация |
| Первичные тесты (рис. 2.46–2.48) | по одному тексту из тестовой выборки на рубрику | выполнено для первого текста тестовой выборки каждой рубрики, плюс новые тексты |
| Код методички | см. таблицу «Отличия кода от методички» | пути Windows, порядок `listdir`, порядок создания `Data_prepared`, `get_feature_names` |
| Кросс-валидация (в методичке нет) | — | accuracy $0{,}867$ / $0{,}889$ / $0{,}867$ для трёх видов очистки: различие в один текст из 45, лемматизация accuracy не повышает |
| Словарь TF-IDF | не приводится | 1430 лемм, векторы нормированы (норма 1,0) |
| Телеграмм-бот | не приводится | `Telethon` через MTProxy; бот использует общую модель по рубрикам из `Data` и `Data_selfwork`, проверочная ячейка сверяет ответы бота с моделью ноутбука по всем обучающим статьям |
| Самостоятельная работа, п. 1 | любые три категории, не менее 15 статей в рубрике | три рубрики из корпуса Lenta.ru; конвейер собран в функции `build_text_classifier` и повторяет основную часть (очистка, TF-IDF, KNN с `k = 4`); итоговые метрики, слова с наибольшим весом, кросс-валидация и ошибки выводятся ячейками после неё |
